<a href="https://colab.research.google.com/github/sandraqonita/sandraqonita-AnalyticsEngineering/blob/main/Copy_of_AE_test_task_C_customers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Analytics Engineering: timed test

## **PLEASE FIRST SAVE A COPY OF THIS FILE! GO TO FILE > SAVE A COPY IN DRIVE.**
### This will mean that your work auto-saves and prevents any issues if you have internet or browser issues. Remember to work on the saved copy. It will be titled "Copy of AE_test_task ..."
<br>

## Task C: Customer payments

## How this test works

- You have **55 minutes to complete the work** with an additional 5 minutes at the end to upload the work. However, there is no end-point as such, you are completing part of what would likely be a longer task.
- This is individual work - please do not confer with anyone inside or outside the room.
- You may use **any AI tool** (for example, Gemini in Colab - which is probably the quickest route). You are **not marked on the quality of the code** it produces. You are marked on how well you control the work: what you look at, what you check, what you reject or correct and why, and what you could not verify.
- Therefore the quality of the AI used has no bearing on your marks, as you are **not marked on the quality of the code** it produces.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make. Notes written afterwards are usually vaguer. If you are using Gemini as in built in the Notebook the prompts are **not** automatically saved - you still need to paste them into section 5 or alongside your code, **along with the checks you determine to make sure the AI code works**.
- Unfinished work is fine if your checking and reasoning are clear. Say what you did not get to.
- Please do not discuss this task with anyone who has not yet sat the test. Other groups have different briefs.

## Rubric

| Element | Marks |
| --- | --- |
| Question one | 100% |
| | |
| Application of analytics engineering techniques: applying methods from the module such as data cleaning, joining, etc. | 35% |
| Careful execution of code (whether generated by AI, from module resources (e.g. GitHub) or memory) and checking the code has worked.  | 35% |
| Analysis and interpretation of results: what do your checks show about the data and what steps should come next? | 30% |


<br>

## The data

You have eight tables from a real Brazilian online marketplace (Olist), covering orders from 2016 to 2018. A geolocation table exists in the full dataset but is not included in this test.

![](https://drive.google.com/uc?export=view&id=10zwOXJ1e9IpT58rhvxgeMqaUZXiqYzDJ)

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | The core table: one record per order, with status and timestamps |
| `order_items` | The items inside each order |
| `payments` | How each order was paid |
| `reviews` | Customer reviews of orders |
| `customers` | Customer details and location |
| `products` | Product details, including category |
| `sellers` | Seller details and location |
| `category_translation` | Maps product category names to English |

**How the tables connect**

- `orders` to `order_items`, `payments` and `reviews` on `order_id`
- `orders` to `customers` on `customer_id`
- `order_items` to `products` on `product_id`
- `order_items` to `sellers` on `seller_id`
- `category_translation` to `products` on the product category name

Check the actual column names in the next cell before you write any joins.


In [1]:
# ---- Setup: run this cell first ----
import glob
import os

import duckdb
import pandas as pd

# Download the files for the test
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testsession/brazilian-ecommerce.zip"    # link to a .zip containing the CSV files

DATA_DIR = "/content/olist"   # upload the CSV files here using the Files panel
  # Download only if no CSV files are there yet. The CSVs may sit inside a folder
  # within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/olist.zip")
    zipfile.ZipFile("/content/olist.zip").extractall(DATA_DIR)

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask your invigilator for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 99,441 rows, 8 columns
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
order_items: 112,650 rows, 7 columns
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
payments: 103,886 rows, 5 columns
    ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
reviews: 89,999 rows, 7 columns
    ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
customers: 99,441 rows, 5 columns
    ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
products: 32,951 rows, 9 columns
    ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'produc

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, c.customer_id
    FROM orders AS o
    JOIN customers AS c ON o.customer_id = c.customer_id
    LIMIT 5
"""))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


,n_rows,n_distinct_orders
0,99441,99441


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["customers"], on="customer_id", how="inner")
display(joined[["order_id", "customer_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


(99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


,order_id,customer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c


99441 different customer_id values in orders


## Your brief

**From:** Finance

> We want to understand how customers pay. For each month, show the share of orders and the share of revenue by payment type, and the average number of instalments for credit card payments. Can we trust that the shares add up?

**What your final table should look like**

One table with columns for month, payment type, number of orders, revenue, share of month’s orders, share of month’s revenue, average instalments.

**Decisions you must make and state**

- An order can have several payment rows (vouchers combined with a card, for example). How will you assign a payment type to an order, or will you count at payment level instead?
- Which orders count, and what does "revenue" mean here?

**If you have time**

Show how many customers placed a second order within 90 days of their first.


## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Strategy: check what one row means in orders and payments, and whether order_id and payments repeat, before linking the two tables.

**What each table holds (to be checked):**
- orders: one row for each order, with its status and the date it was placed.
- payments: one row for each payment. If an order was paid in more than one
  way, for example part voucher and part card, it has more than one row. Each
  row shows the payment type, the amount and the number of instalments.
- customers: one row for each customer ID. I only need this for the extra
  question about repeat customers, where I'll check whether one person can
  have more than one customer ID.
- I don't need order_items, reviews, products, sellers or
  category_translation for this question, so I'll leave them out.

What my final table will look like: one row for each month and payment type. This includes showing the number of orders, the revenue, that payment type's share of
the month's orders and revenue, and the average number of instalments for credit cards. The month comes from orders; everything else comes from payments. So I need to link payments to orders using order_id. Each payment belongs to only one order, so linking them should not add any rows.

What I need to watch out for: some orders were paid in more than one way. If I count orders by payment type, an order paid by voucher and card would be counted twice, so the order shares for a month could add up to more than 100%. Revenue shares should still add up to 100%, because each payment amount is only counted once. Checking that both shares add up to 100% each month is how I'll answer "can we trust that the shares add up?"

\What I expect to find: orders with more than one payment, a payment type that needs checking, orders that weren't delivered, dates stored as text, and maybe some orders with no payment or payments with no matching order. I'll check each of these later.

**1. First look at each table**

prompt: Go through every table in tables and, for each one, show its name, how many rows it has and its first three rows

I expect: Tables match the brief's description; row counts are sensible

Table Name: orders
Number of rows: 99,441
First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


--------------------------------------------------
Table Name: order_items
Number of rows: 112,650
First 5 rows:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


--------------------------------------------------
Table Name: payments
Number of rows: 103,886
First 5 rows:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


... (7 outputs left out)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


--------------------------------------------------
Table Name: category_translation
Number of rows: 71
First 5 rows:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


--------------------------------------------------


What I saw: the tables look sensible, with category has the smallest size (71 rows), followed by sellers (3,095 rows)

**Missing value**

Prompt: for each table in tables, count the missing values in every column and print the results

I expect: a list of fields with a count of missing values each; for payments and customers, hopefully all zero

Table Name: orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
--------------------------------------------------
Table Name: order_items
No missing values
--------------------------------------------------
Table Name: payments
No missing values
--------------------------------------------------
Table Name: reviews
review_comment_title      79404
review_comment_message    52429
dtype: int64
--------------------------------------------------
Table Name: customers
No missing values
--------------------------------------------------
Table Name: products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
--------------------------------------------------
Table Name: sellers
No m

What I saw: payments, customers, order_items, sellers, and category_translations have no missing values. Orders and product have missing values. Review has most missing values (make sense). Will not investigate further for products and review

**Duplicates and uniqueness**

prompt: For orders, payments and customers, print the total number of rows, the number of exact duplicate rows, and the number of unique ids. The ids should be: order_id for orders; order_id for payments, and also order_id and
payment_sequential together for payments; customer_id and customer_unique_id for customers.

I expect: Ideally zero duplicates, and every order_id and customer_id are unique

=== ORDERS ===
Total rows: 99,441
Exact duplicate rows: 0
Unique order_id: 99,441

=== PAYMENTS ===
Total rows: 103,886
Exact duplicate rows: 0
Unique order_id: 99,440
Unique (order_id, payment_sequential): True (Unique count: 103,886)

=== CUSTOMERS ===
Total rows: 99,441
Exact duplicate rows: 0
Unique customer_id: 99,441
Unique customer_unique_id: 96,096


What I saw: no exact duplicates rows for all; order_id x payment has unique counts 103,886

**Labels and categories**

prompt: For orders, show how many rows there are for each order_status, including missing values. For payments, show how many rows and how many different orders (order_id) there are for each payment_type, including missing values. Then show how many orders have exactly 1, 2, 3 or more payment rows, and list the most common combinations of payment types within one order (for example credit_card + voucher)

I expect: order_status to have a few values, with most orders "delivered" and smaller numbers of shipped, canceled and others. payment_type to have about five values (credit_card, boleto, voucher, debit_card and possibly
not_defined), with credit_card the largest. Rows should be more than orders for at least voucher, since orders pay with several vouchers. Most orders should have 1 payment row, with a smaller number having 2 or more, mostly
card + voucher. Spellings and capitals may also be inconsistent.


=== Row count by order_status (including missing values) ===
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


=== Row counts and distinct orders by payment_type ===
              row_count  distinct_orders
payment_type                            
boleto            19784            19784
credit_card       76795            76505
debit_card         1529             1528
not_defined           3                3
voucher            5775             3866


=== Distribution of payment rows per order ===
1 payment row             96479
2 payment rows             2382
3 or more payment rows      579
dtype: int64


=== Most common combinations of payment types within one order ===
payment_type
(credit_card,)               74259
(boleto,)                    19784
(credit_card, voucher)        2245
(voucher,)                    1621
(debit_c

What I see: Order status: 8 statuses, no missing values or messy spellings. They add up
to 99,441, matching the orders table. About 97% (96,478) were delivered. The rest are shipped, canceled, unavailable, invoiced, processing, created or approved. (need to check if have more time)

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table represents, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

silver_orders (one row per order, with status and purchase month) [= rows in orders, 99,441]; silver_payments (one row per payment, duplicates removed)
[= rows in payments, 103,886, unless duplicates are found];
silver_order_payments (one row per order: total paid, main payment type, and credit card instalments) [= orders with a payment, 99,440]; silver_order joins silver_orders with silver_order_payments [= rows in orders, 99,441; the one order with no payment stays in, with an empty payment type]: "The row count after every join must not change"; gold_monthly_payments grouped from silver_order by month and payment type, with order shares and revenue shares that add up to 100% for each month.

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

**silver_orders**

prompt: from the orders table, please convert order_purchase_timestamp to a
datetime object as a new column called purchase_ts. Create a field called order_month that shows the first day of the month of purchase_ts. Keep order_id, customer_id, order_status, purchase_ts and order_month. Print the data type of purchase_ts before and after, the top 5 rows, and the number of
rows. Then save this as silver_orders, a new table

I expect: order_purchase_timestamp before and datetime after, order_month always the 1st of a month, and 99,441 rows, the same as orders.

Data type of order_purchase_timestamp before: object
Data type of purchase_ts after: datetime64[ns]
Total rows in silver_orders: 99,441


,order_id,customer_id,order_status,purchase_ts,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-01
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-01
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-01
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-01
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-01


What I see: purchase time changed from text (object) to a date (datetime64), as expected. 99,441 rows, the same as orders, so no orders were lost. order_month is always the 1st of the month in the sample rows, an order on 2 October 2017 has order_month 2017-10-01.

**silver_payments**

prompt: create a table called silver_payments from the payments table. Drop exact duplicate rows, and print the number of rows before and after. Check that order_id and payment_sequential together are unique. Show the value counts of payment_type and the min and max of payment_value and payment_installments so I can check.

I expect: 103,886 rows before, and the same after unless duplicates are
found. order_id + payment_sequential unique. 5 payment types. no negative
payment_value. instalments of 1 or more for credit card.

Rows before: 103,886
Rows after: 103,886
Removed 0 exact duplicate rows.

Is (order_id, payment_sequential) unique? True

=== Payment Type Value Counts ===
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

=== Statistics check ===
     payment_value  payment_installments
min           0.00                     0
max       13664.08                    24


What I see: 103,886 rows before and after, so there were no exact duplicates to remove. order_id + payment_sequential is unique, so each row is one payment. The 5 payment types have the same counts as in my profiling.
Two things I didn't expect: the smallest payment_value is 0.00, and the smallest number of instalments is 0, but I expected at least 1 for credit card. I haven't checked which rows these are; I've noted this in Step 6.

**silver_order_payments**

prompt: please create a table called silver_order_payments from
silver_payments, with one row per order_id. Include the sum of payment_value as total_paid; the number of payment rows as n_payments; the number of
different payment types as n_payment_types; main_payment_type as [the
payment_type with the largest total payment_value in that order]; and cc_installments as the maximum payment_installments of the credit_card rows
in that order (empty if there are none). Print the first five rows, the number of rows, and the value counts of main_payment_type

I expect: 99,440 rows, one per order with a payment, and order_id unique. Every order has exactly one main payment type, so the counts by main_payment_type add up to 99,440.

Number of rows in silver_order_payments: 99,440

=== Value counts of main_payment_type ===
main_payment_type
credit_card    74935
boleto         19784
voucher         3191
debit_card      1527
not_defined        3
Name: count, dtype: int64

=== First 5 rows of silver_order_payments ===


,order_id,total_paid,n_payments,n_payment_types,main_payment_type,cc_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,credit_card,2.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,credit_card,3.0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,credit_card,5.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,credit_card,2.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,credit_card,3.0


What I see: 99,440 rows, one for each order with a payment, as expected. The main payment type counts add up to 99,440, so every order has exactly one type. This matches my profiling:
- boleto (19,784) and not_defined (3) are unchanged, as these were never mixed with another type.
- Of the 2,245 orders paid by credit card plus voucher, 1,570 now count as voucher (the voucher was the larger amount) and 675 as credit card.
- The one credit card plus debit card order counts as credit card, so debit card is 1,527.
The 3 not_defined orders are kept as their own type.

**silver_order**

prompt: create a table called silver_order. This should be a left join on silver_orders with silver_order_payments, on order_id. Include all columns
from silver_orders, and total_paid, n_payments, n_payment_types, main_payment_type and cc_installments from silver_order_payments. Print the
number of rows in silver_order and in orders so we can check they are the same, and how many rows have no main_payment_type

I expect: 99,441 rows in both, and 1 row with no main_payment_type (the order
with no payment).

Number of rows in silver_order: 99,441
Number of rows in orders: 99,441
Are they identical? True
Number of rows with no main_payment_type: 1


What I see: 99,441 rows in silver_order and in orders, so the join didn't add or drop any orders. 1 row has no main_payment_type: the order with no payment, expected

**Gold table**

prompt: create a table called gold_monthly_payments from silver_order, keeping only orders where order_status is [e.g. not 'canceled' or 'unavailable'] and main_payment_type is not empty. Group by order_month and main_payment_type. Include the count of orders as n_orders; the sum of total_paid rounded to two decimal places as revenue; n_orders divided by the
month's total orders as share_of_orders; revenue divided by the month's total revenue as share_of_revenue; and the average of cc_installments as avg_installments (credit card only). Print the first 10 rows, the total number of rows, and for each month the sum of share_of_orders and
share_of_revenue

I expect: one row per month and payment type, and both shares adding up to 1.00 (100%) in every month.

Total number of rows in gold_monthly_payments: 87

First 10 rows:


,order_month,main_payment_type,n_orders,revenue,avg_installments,share_of_orders,share_of_revenue
0,2016-09-01,credit_card,1,136.23,1.000000,1.000000,1.000000
1,2016-10-01,boleto,60,9226.91,NaN,0.204778,0.178617
2,2016-10-01,credit_card,222,41213.61,4.481982,0.757679,0.797824
3,2016-10-01,debit_card,2,241.73,NaN,0.006826,0.004679
4,2016-10-01,voucher,9,975.28,NaN,0.030717,0.018880
5,2016-12-01,credit_card,1,19.62,1.000000,1.000000,1.000000
6,2017-01-01,boleto,193,23685.11,NaN,0.245235,0.172875
7,2017-01-01,credit_card,557,107936.84,3.854578,0.707751,0.787821
8,2017-01-01,debit_card,9,743.53,NaN,0.011436,0.005427
9,2017-01-01,voucher,28,4641.28,NaN,0.035578,0.033876



=== Verification: Sum of shares by month ===


,share_of_orders,share_of_revenue
order_month,,
2016-09-01,1.0,1.0
2016-10-01,1.0,1.0
2016-12-01,1.0,1.0
2017-01-01,1.0,1.0
2017-02-01,1.0,1.0
2017-03-01,1.0,1.0
2017-04-01,1.0,1.0
2017-05-01,1.0,1.0
2017-06-01,1.0,1.0


What I see: 87 rows, one per month and payment type. That's fewer than the maximum of 120 (24 months x 5 types), because not every type is used in every month. Both shares add up to 1.0 (100%) in every month, so the shares can be trusted.

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

couldn't do it

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. If applicable, include at least one example where the AI was wrong or needed correcting.*

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

**What I could not verify**

- Which order statuses the gold table leaves out.
- Why some payments are 0.00 or have 0 instalments.
- What the 3 "not_defined" payments are.
- The time zone. Orders near midnight could fall in the wrong month.
- Whether amount paid matches item prices plus freight.
- The first and last months, which have very few orders.
- Whether gold totals match silver_order.
- The repeat-customer question. I didn't get to it.

**What I would do next**

- Look at the odd payment rows.
- Compare gold totals with silver_order, and payments with order_items.
- Build the repeat-customer table.